In [1]:

from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(len(v))
print(v[0])


384
-0.020582034180917443


In [3]:
from embedder import Embedder

embedder = Embedder()

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
print(len(documents))


72


In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
query = "How does approximate nearest neighbor search work?"

query_vector = embedder.encode(query)

In [8]:
print(query_vector.shape)

(384,)


In [9]:
for doc in documents:
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        target_doc = doc
        break

In [10]:
target_doc["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [11]:
doc_vector = embedder.encode(target_doc["content"])

In [12]:
import numpy as np

similarity = np.dot(query_vector, doc_vector)

print(similarity)

0.36107008472347096


In [13]:
doc_vector = embedder.encode(target_doc["content"])

In [14]:
import numpy as np

similarity = np.dot(query_vector, doc_vector)

print(similarity)

0.36107008472347096


In [15]:
import numpy as np

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

doc_v = embedder.encode(doc["content"])

similarity = np.dot(v, doc_v)

print(similarity)

0.36107008472347096


In [16]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

In [18]:
chunk_vectors = embedder.encode_batch(
    [chunk["content"] for chunk in chunks]
)

In [19]:
type(chunk_vectors)
chunk_vectors.shape

(295, 384)

In [22]:
X = chunk_vectors

In [20]:
query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

In [23]:
scores = X.dot(v)

In [24]:
scores.shape

(295,)

In [25]:
best = scores.argmax()

print(best)

94


In [26]:
chunks[best]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [30]:
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(
    vectors=X,
    payload=chunks
)

In [28]:
import inspect
from minsearch import VectorSearch

print(inspect.signature(VectorSearch))

(keyword_fields=None, numeric_fields=None, date_fields=None)


In [31]:
query = "What metric do we use to evaluate a search engine?"
q = embedder.encode(query)

results = vindex.search(
    query_vector=q,
    num_results=5
)

print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


In [32]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [33]:
query = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(
    query,
    num_results=5
)

text_files = [r["filename"] for r in text_results]
text_files

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [34]:
q = embedder.encode(query)

vector_results = vindex.search(
    query_vector=q,
    num_results=5
)

vector_files = [r["filename"] for r in vector_results]
vector_files

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [36]:
set(vector_files) - set(text_files)

{'02-vector-search/lessons/08-pgvector.md'}

In [37]:
query = "How do I give the model access to tools?"

# vector search
q = embedder.encode(query)

vector_results = vindex.search(
    query_vector=q,
    num_results=5
)

# text search
text_results = text_index.search(
    query,
    num_results=5
)

In [38]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [39]:
results = rrf([vector_results, text_results])

print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
